In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

# 1. 운영체제(OS)에 따른 폰트 자동 설정
system_name = platform.system()

if system_name == 'Windows':
    # 윈도우: 맑은 고딕 (Malgun Gothic)
    plt.rc('font', family='Malgun Gothic')
elif system_name == 'Darwin':
    # 맥(Mac): 애플고딕 (AppleGothic)
    plt.rc('font', family='AppleGothic')
else:
    # 리눅스(Colab 등): 나눔고딕 설치 필요 (기본적으로는 없을 수 있음)
    # Colab 사용 시 별도 설치 명령어가 필요합니다.
    print("Linux/Colab 환경입니다. 폰트 수동 설정이 필요할 수 있습니다.")
    # 임시로 나눔고딕 설정 시도
    plt.rc('font', family='NanumGothic')

# 2. 마이너스 기호(-) 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

print(f"✅ 폰트 설정 완료: {plt.rcParams['font.family']}")

In [ ]:
import os
import glob
import json
import pandas as pd
from tqdm import tqdm

# ==========================================
# 설정: 뉴스 데이터 경로 지정
# ==========================================
# 라벨링 데이터 경로 (여기에 본문과 요약이 같이 있는 경우가 많습니다)
BASE_DIR = "data/022.요약문_및_레포트_생성_데이터/01.데이터/1.Training/라벨링데이터"
TARGET_Category = "01.news_r" # 뉴스 데이터만 지정

OUTPUT_FILE = "news_data_processed.csv"
MAX_SAMPLES = 100 # 테스트용으로 100개만 추출 (전체 하려면 None으로 변경)

def load_news_data():
    print(f"📂 뉴스 데이터 로드 시작: {TARGET_Category}")
    
    # 01.news_r 폴더 밑의 모든 JSON 파일 검색
    search_path = os.path.join(BASE_DIR, TARGET_Category, "**", "*.json")
    json_files = glob.glob(search_path, recursive=True)
    
    print(f"   -> 총 {len(json_files)}개의 파일을 발견했습니다.")
    
    data_list = []
    
    # 파일 읽기
    for f_path in tqdm(json_files[:MAX_SAMPLES] if MAX_SAMPLES else json_files):
        try:
            with open(f_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
                # 데이터 구조에 따라 파싱 (제공해주신 train_model 참조)
                # 보통 Meta(Refine)에 본문, Annotation에 요약이 있음
                if 'Meta(Refine)' in data and 'Annotation' in data:
                    passage = data['Meta(Refine)'].get('passage', "")
                    
                    # 요약문 추출 (summary1 우선)
                    ann = data['Annotation']
                    summary = ""
                    if isinstance(ann, dict):
                        summary = ann.get('summary1', "")
                    elif isinstance(ann, list): # 리스트인 경우
                        summary = ann[0].get('summary1', "")
                    
                    if passage and summary:
                        data_list.append({
                            "filename": os.path.basename(f_path),
                            "original_text": passage,
                            "ground_truth": summary
                        })
        except Exception as e:
            continue
            
    df = pd.DataFrame(data_list)
    print(f"✅ 데이터 로드 완료! 총 {len(df)}개")
    
    # CSV 저장
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"💾 {OUTPUT_FILE} 저장 완료")
    return df

if __name__ == "__main__":
    load_news_data()

In [ ]:
import pandas as pd
import torch
from tqdm import tqdm
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig,
    pipeline
)

# ==========================================
# 설정 (Configuration)
# ==========================================
INPUT_CSV = "news_data_processed.csv"  # 위에서 만든 파일
OUTPUT_CSV = "model_comparison_result.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 4-bit 양자화 설정 (속도 UP, 메모리 DOWN)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# ==========================================
# 1. 모델 로드 함수들
# ==========================================

def load_bart():
    print("🔹 Loading KoBART...")
    model_name = "gogamza/kobart-summarization"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE)
    return tokenizer, model

def load_polyglot():
    print("🔹 Loading Polyglot-Ko (4-bit)...")
    # 메모리에 맞게 1.3b 또는 5.8b 선택
    model_name = "EleutherAI/polyglot-ko-1.3b" 
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        quantization_config=bnb_config, # 양자화 적용
        device_map="auto"
    )
    return tokenizer, model

def load_llama():
    print("🔹 Loading LLaMA-3-Ko (4-bit)...")
    # 한국어 성능이 좋은 Llama-3 파생 모델
    model_name = "beomi/Llama-3-Open-Ko-8B"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config, # 양자화 적용
        device_map="auto"
    )
    return tokenizer, model

# ==========================================
# 2. 추론 함수 정의
# ==========================================

def generate_summary_bart(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True).to(DEVICE)
    summary_ids = model.generate(
        inputs["input_ids"], 
        max_length=150, 
        min_length=30, 
        length_penalty=2.0, 
        num_beams=4
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

def generate_summary_causal(text, tokenizer, model, model_type="polyglot"):
    # Causal LM(Polyglot, LLaMA)은 프롬프트가 중요합니다.
    if model_type == "polyglot":
        prompt = f"다음 뉴스 기사를 한 문단으로 요약해줘.\n\n기사: {text}\n\n요약:"
    else: # LLaMA-3 스타일
        prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n다음 글을 핵심만 요약해줘:\n{text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
    
    inputs = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(DEVICE)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 프롬프트 부분 제거하고 결과만 추출
    return generated.replace(prompt, "").split("요약:")[-1].strip()

# ==========================================
# 3. 메인 실행
# ==========================================
def main():
    # 1. 데이터 로드
    if not os.path.exists(INPUT_CSV):
        print("❌ 처리된 데이터 파일이 없습니다. Step 1을 먼저 실행하세요.")
        return
    
    df = pd.read_csv(INPUT_CSV)
    print(f"📊 데이터 {len(df)}개 로드 완료. 비교 시작!")

    results = []

    # 2. 모델별 순차 실행 (메모리 부족 방지를 위해 하나씩 로드/해제 추천하지만, 4090은 4bit시 다 올라갈 수도 있음)
    # 여기서는 안전하게 하나씩 수행합니다.
    
    # --- BART ---
    tok_bart, mod_bart = load_bart()
    bart_sums = []
    print("🚀 BART 추론 중...")
    for text in tqdm(df['original_text']):
        bart_sums.append(generate_summary_bart(text, tok_bart, mod_bart))
    del mod_bart, tok_bart
    torch.cuda.empty_cache()

    # --- Polyglot ---
    tok_poly, mod_poly = load_polyglot()
    poly_sums = []
    print("🚀 Polyglot 추론 중...")
    for text in tqdm(df['original_text']):
        poly_sums.append(generate_summary_causal(text, tok_poly, mod_poly, "polyglot"))
    del mod_poly, tok_poly
    torch.cuda.empty_cache()

    # --- LLaMA ---
    tok_llama, mod_llama = load_llama()
    llama_sums = []
    print("🚀 LLaMA 추론 중...")
    for text in tqdm(df['original_text']):
        # LLaMA는 텍스트가 너무 길면 잘릴 수 있으므로 길이 제한
        truncated_text = text[:1500] 
        llama_sums.append(generate_summary_causal(truncated_text, tok_llama, mod_llama, "llama"))
    del mod_llama, tok_llama
    torch.cuda.empty_cache()

    # 3. 결과 저장
    df['BART_Summary'] = bart_sums
    df['Polyglot_Summary'] = poly_sums
    df['LLaMA_Summary'] = llama_sums
    
    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    print(f"\n✨ 모든 작업 완료! 결과 파일: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import torch
import re
from tqdm import tqdm
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig
)
import os

# ==========================================
# 1. 설정 (Configuration)
# ==========================================
# ★ 중요: 학습시킨 모델 경로로 변경해주세요!
BART_MODEL_PATH = "gogamza/kobart-summarization"  # 예: checkpoint-42500 또는 final_model 폴더
\
INPUT_CSV = "news_data_processed.csv"
OUTPUT_CSV = "model_comparison_result_v2.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 2. [중요] 후처리 함수 (요약모델.ipynb에서 가져옴)
# ==========================================
def tidy_korean_summary(text: str) -> str:
    if not isinstance(text, str): return ""
    t = re.sub(r"\s+", " ", text).strip()
    if t and t[-1] not in ".!?…。\"'”’":
        t += "."
    return t

def drop_incomplete_sentences(text: str) -> str:
    if not text or str(text).strip() == "":
        return ""
    sentences = re.split(r'(?<=[\.\?\!。\!\?])\s*', text)
    sentences = [s.strip() for s in sentences if s and s.strip()]
    if len(sentences) == 0:
        return ""
    if len(sentences) == 1:
        last = sentences[0]
        # 너무 짧거나 접속사로 끝나는 문장 제거
        if len(last.split()) < 3 or re.search(r'(며$|고$|하며$|으로$|에게$|서$|로$|에$|에 대해$|한다$|된다$)', last):
            return ""
        return last
    
    last = sentences[-1]
    # 마지막 문장이 불완전하면 제거
    if (len(last.split()) < 4) or (not re.search(r'[\.\?\!。\!\?:;]$', last)) or re.search(r'(며$|고$|하며$|으로$|에게$|서$|로$|에$|에 대해$|한다$|된다$|것이다$)', last):
        sentences = sentences[:-1]
    
    cleaned = " ".join(sentences).strip()
    return cleaned

def post_process(text):
    """위 두 함수를 한 번에 적용"""
    text = tidy_korean_summary(text)
    text = drop_incomplete_sentences(text)
    return text

# ==========================================
# 3. 모델 로드 및 생성 함수
# ==========================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

def generate_bart(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True).to(DEVICE)
    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"], 
            max_length=150, 
            min_length=40,       # 노트북 설정 반영
            length_penalty=1.2,  # 노트북 설정 반영
            num_beams=5,         # 노트북 설정 반영
            no_repeat_ngram_size=3,
            early_stopping=True
        )
    raw_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return post_process(raw_summary)

def generate_polyglot(text, tokenizer, model):
    # Polyglot은 Zero-shot보다 One-shot(예시 하나 주기)이 훨씬 잘 됩니다.
    # 하지만 여기선 간단히 명확한 지시문으로 변경합니다.
    prompt = f"다음 뉴스 기사의 핵심 내용을 3문장 이내로 요약해.\n\n기사: {text}\n\n요약:"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=150,
            do_sample=True,
            temperature=0.5, # 창의성 낮춤 (정확도 위주)
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 프롬프트 제거 로직 강화
    if "요약:" in decoded:
        result = decoded.split("요약:")[-1].strip()
    else:
        # 혹시 요약: 태그가 잘렸을 경우 프롬프트 길이만큼 잘라냄
        result = decoded[len(prompt):].strip()
        
    return post_process(result)

def generate_llama(text, tokenizer, model):
    # LLaMA-3 Instruct 포맷 준수
    prompt = (
        f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
        f"다음 기사를 한국어로 3문장 요약해줘:\n{text}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=150,
            do_sample=True,
            temperature=0.3, # LLaMA는 0.2~0.3이 요약에 좋음
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # LLaMA-3 특수 토큰 및 프롬프트 제거
    # assistant 헤더 뒤의 내용만 가져옴
    if "assistant" in decoded:
        result = decoded.split("assistant")[-1].strip()
    else:
        result = decoded
        
    return post_process(result)

# ==========================================
# 4. 메인 실행 루프
# ==========================================
def main():
    if not os.path.exists(INPUT_CSV):
        print("❌ CSV 파일이 없습니다. Step 1을 먼저 실행하세요.")
        return
    
    df = pd.read_csv(INPUT_CSV)
    
    # --- 1. BART (Fine-tuned) ---
    print(f"🔹 BART 로드 중... ({BART_MODEL_PATH})")
    try:
        tok_bart = AutoTokenizer.from_pretrained(BART_MODEL_PATH)
        mod_bart = AutoModelForSeq2SeqLM.from_pretrained(BART_MODEL_PATH).to(DEVICE)
        
        bart_sums = []
        for text in tqdm(df['original_text'], desc="BART Generating"):
            bart_sums.append(generate_bart(text, tok_bart, mod_bart))
            
        df['BART_Summary'] = bart_sums
        del mod_bart, tok_bart
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"❌ BART 실행 실패: {e}")

    # --- 2. Polyglot ---
    print("🔹 Polyglot 로드 중...")
    try:
        # 1.3b가 너무 작아서 성능이 안나오면 5.8b 추천 (3090이면 5.8b 4bit 충분히 가능)
        model_name = "EleutherAI/polyglot-ko-5.8b" 
        tok_poly = AutoTokenizer.from_pretrained(model_name)
        mod_poly = AutoModelForCausalLM.from_pretrained(
            model_name, 
            quantization_config=bnb_config,
            device_map="auto"
        )
        
        poly_sums = []
        for text in tqdm(df['original_text'], desc="Polyglot Generating"):
            poly_sums.append(generate_polyglot(text, tok_poly, mod_poly))
            
        df['Polyglot_Summary'] = poly_sums
        del mod_poly, tok_poly
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"❌ Polyglot 실행 실패: {e}")

    # --- 3. LLaMA ---
    print("🔹 LLaMA 로드 중...")
    try:
        model_name = "beomi/Llama-3-Open-Ko-8B"
        tok_llama = AutoTokenizer.from_pretrained(model_name)
        mod_llama = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto"
        )
        
        llama_sums = []
        for text in tqdm(df['original_text'], desc="LLaMA Generating"):
            # 입력 길이 제한 (메모리 보호)
            text_input = text[:1500]
            llama_sums.append(generate_llama(text_input, tok_llama, mod_llama))
            
        df['LLaMA_Summary'] = llama_sums
        del mod_llama, tok_llama
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"❌ LLaMA 실행 실패: {e}")

    # 저장
    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    print(f"\n✨ 완료! 결과 파일: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import torch
import re
import os
import gc
from tqdm import tqdm
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig
)

# ==========================================
# 1. 설정 (Configuration)
# ==========================================
# ★ 중요: 사용자가 학습시킨 BART 모델 경로 (없으면 폴더명 확인 필요)
BART_CHECKPOINT = "gogamza/kobart-summarization" 

# 비교할 모델들 (RTX 3090 24GB 기준 최적 세팅)
MODELS = {
    "BART": BART_CHECKPOINT,
    "Polyglot": "EleutherAI/polyglot-ko-5.8b",  # 1.3b보다 5.8b 추천 (3090 가능)
    "LLaMA": "beomi/Llama-3-Open-Ko-8B"
}

INPUT_CSV = "news_data_processed.csv"       # 1단계에서 만든 데이터 파일
OUTPUT_CSV = "model_comparison_final.csv"   # 최종 결과 파일
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 4-bit 양자화 설정 (메모리 절약 + 속도 향상)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# ==========================================
# 2. 강력한 전처리 및 후처리 함수
# ==========================================
def deep_clean_text(text):
    """모델 입력 전, 텍스트의 노이즈(줄바꿈, 끊긴 숫자 등)를 제거"""
    if not isinstance(text, str): return ""
    
    # 1. '1. \n 6' 처럼 숫자가 끊긴 경우 복구 -> '1.6'
    text = re.sub(r'(\d+)\.\s+[\n\r]*(\d+)', r'\1.\2', text)
    
    # 2. 불필요한 줄바꿈과 다중 공백 제거
    text = re.sub(r'[\n\r]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    
    # 3. 특수문자 노이즈 제거 (선택적)
    text = text.replace("quot;", "'").replace("&amp;", "&")
    
    return text.strip()

def post_process_summary(text):
    """모델이 생성한 요약문의 완성도를 높임 (문장 끊김 제거)"""
    if not text: return ""
    
    # 1. 불필요한 공백 제거
    text = re.sub(r"\s+", " ", text).strip()
    
    # 2. 문장 단위로 분리
    sentences = re.split(r'(?<=[\.\?\!])\s*', text)
    valid_sentences = []
    
    for s in sentences:
        s = s.strip()
        if not s: continue
        # ~며, ~고 등으로 끝나는 불완전 문장 제거
        if re.search(r'(며|고|하며|으로|에게|서|로|에|에 대해|한다|된다|것이다)$', s):
            continue
        valid_sentences.append(s)
            
    # 3. 마지막 문장이 마침표로 안 끝나면 제거
    if valid_sentences and not re.search(r'[\.\?\!]$', valid_sentences[-1]):
        valid_sentences.pop()
        
    return " ".join(valid_sentences)

# ==========================================
# 3. 모델별 생성 함수 (최적 파라미터 적용)
# ==========================================

def run_bart(df, model_path):
    print(f"\n🚀 [1/3] BART 모델 로드 중... ({model_path})")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE)
        
        summaries = []
        for text in tqdm(df['cleaned_text'], desc="BART Inference"):
            inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True).to(DEVICE)
            
            with torch.no_grad():
                summary_ids = model.generate(
                    inputs["input_ids"],
                    max_length=150,
                    min_length=30,
                    num_beams=4,
                    no_repeat_ngram_size=3,  # 3-gram 반복 방지
                    repetition_penalty=1.5,  # ★ 반복 억제 강화 (1.2 -> 1.5)
                    length_penalty=1.0,
                    early_stopping=True
                )
            decoded = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
            summaries.append(post_process_summary(decoded))
            
        del model, tokenizer
        torch.cuda.empty_cache()
        return summaries
    except Exception as e:
        print(f"❌ BART 에러: {e}")
        return ["Error"] * len(df)

# ==========================================
# [수정] 강력해진 전처리/후처리 및 생성 로직
# ==========================================

def clean_llama_result(text):
    """LLaMA 결과에서 불필요한 태그와 형식을 제거"""
    # 1. '요약:', '결론:' 같은 접두어 제거
    text = re.sub(r"^(요약|결론|Summary|Note)[:\s]*", "", text, flags=re.IGNORECASE)
    
    # 2. 리스트 형식(1. 2. 3.) 제거하고 문장으로 합치기
    # 예: "1. 발언했다. 2. 주장했다." -> "발언했다. 주장했다."
    text = re.sub(r"\d+\.\s*", "", text)
    
    return text.strip()

def run_polyglot(df, model_name):
    print(f"\n🚀 [2/3] Polyglot(5.8B) 모델 재시도 중...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        # 5.8B가 자꾸 터지면 1.3B로 낮춰서 테스트 해보시거나, 
        # max_memory 설정을 추가하여 VRAM 할당을 최적화해야 합니다.
        model = AutoModelForCausalLM.from_pretrained(
            model_name, 
            quantization_config=bnb_config, 
            device_map="auto"
        )
        
        summaries = []
        for text in tqdm(df['cleaned_text'], desc="Polyglot Inference"):
            # 프롬프트: 더 명확하게 '문어체'를 요구
            prompt = f"""다음 뉴스 기사를 뉴스 말투로 3문장 이내로 요약해라.
기사: {text}
요약:"""
            
            inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=150,
                    do_sample=True,
                    temperature=0.1,  # 0.2보다 더 낮춰서 확실한 문장만 뱉게 설정
                    repetition_penalty=1.2,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )
            
            decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
            if "요약:" in decoded:
                result = decoded.split("요약:")[-1].strip()
            else:
                result = decoded[len(prompt):].strip()
                
            summaries.append(post_process_summary(result))
            
        return summaries

    except Exception as e:
        # 에러 원인을 출력하여 디버깅
        print(f"\n❌ Polyglot 치명적 에러 발생: {e}")
        import traceback
        traceback.print_exc() # 상세 에러 로그 출력
        return ["Error"] * len(df)

def run_llama(df, model_name):
    print(f"\n🚀 [3/3] LLaMA-3 모델 개선된 프롬프트 적용...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_config, device_map="auto"
        )
        if tokenizer.pad_token_id is None: tokenizer.pad_token_id = tokenizer.eos_token_id
            
        summaries = []
        for text in tqdm(df['cleaned_text'], desc="LLaMA Inference"):
            safe_text = text[:1500] 
            
            # [핵심] 시스템 프롬프트: 말투(Tone)와 형식(Format) 강제
            prompt = (
                f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
                f"당신은 뉴스 편집자입니다. 다음 기사를 '하다'체(문어체)로 간결하게 한 문단으로 요약하세요. "
                f"번호를 매기거나 목록을 만들지 마세요. 인사말이나 '요약:' 같은 사족을 붙이지 마세요.<|eot_id|>"
                f"<|start_header_id|>user<|end_header_id|>\n\n"
                f"기사:\n{safe_text}\n\n위 기사를 요약해.<|eot_id|>"
                f"<|start_header_id|>assistant<|end_header_id|>\n\n"
            )
            
            inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=200, # 길이를 조금 늘려줌
                    do_sample=True,
                    temperature=0.1,    # 사실 위주 생성
                    top_p=0.9,
                    repetition_penalty=1.1,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )
            
            decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
            if "assistant" in decoded:
                result = decoded.split("assistant")[-1].strip()
            else:
                result = decoded
            
            # 추가 클리닝 함수 적용
            result = clean_llama_result(result)
            summaries.append(post_process_summary(result))
            
        return summaries
    except Exception as e:
        print(f"❌ LLaMA 에러: {e}")
        return ["Error"] * len(df)
# ==========================================
# 4. 메인 실행
# ==========================================
def main():
    # 1. 데이터 로드
    if not os.path.exists(INPUT_CSV):
        print(f"❌ 입력 파일({INPUT_CSV})이 없습니다. Step 1 코드를 먼저 실행해주세요.")
        return

    print("📊 데이터 전처리 시작...")
    df = pd.read_csv(INPUT_CSV)
    
    # [핵심] 딥 클리닝 적용
    df['cleaned_text'] = df['original_text'].apply(deep_clean_text)
    
    # 2. 모델별 실행 (순차적)
    # GPU 메모리 관리를 위해 하나씩 로드하고 지웁니다.
    
    # (1) BART (Fine-tuned)
    if os.path.exists(MODELS["BART"]):
        df['BART_Summary'] = run_bart(df, MODELS["BART"])
    else:
        print("⚠️ 학습된 BART 모델 경로가 없어 HuggingFace 기본 모델을 사용합니다.")
        df['BART_Summary'] = run_bart(df, "gogamza/kobart-summarization")
        
    # (2) Polyglot-Ko
    df['Polyglot_Summary'] = run_polyglot(df, MODELS["Polyglot"])
    
    # (3) LLaMA-3-Ko
    df['LLaMA_Summary'] = run_llama(df, MODELS["LLaMA"])
    
    # 3. 결과 저장
    final_columns = ['filename', 'original_text', 'cleaned_text', 'ground_truth', 'BART_Summary', 'Polyglot_Summary', 'LLaMA_Summary']
    # 컬럼이 존재하는지 확인 후 저장
    save_cols = [c for c in final_columns if c in df.columns]
    
    df[save_cols].to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    print(f"\n✨ 모든 작업 완료! 최종 결과: {OUTPUT_CSV}")
    print("   -> 이제 evaluate_results.py 를 실행하여 점수를 확인하세요.")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import torch
import re
import os
import gc
from tqdm import tqdm
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig
)

# ==========================================
# 1. 설정 (Configuration)
# ==========================================
# ★ BART 경로: 사용자님의 학습 모델 경로로 꼭 확인해주세요!
BART_CHECKPOINT = "./results/final_model"

MODELS = {
    "BART": BART_CHECKPOINT,
    "Polyglot": "EleutherAI/polyglot-ko-5.8b",
    # ★ [중요 수정] Base 모델 -> Instruct 모델로 변경 (프롬프트 인식 가능)
    "LLaMA": "beomi/Llama-3-Open-Ko-8B-Instruct-preview"
}

INPUT_CSV = "news_data_processed.csv"
OUTPUT_CSV = "model_comparison_final_v3.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 4-bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# ==========================================
# 2. 전처리 및 후처리 함수
# ==========================================
def deep_clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'(\d+)\.\s+[\n\r]*(\d+)', r'\1.\2', text) # 끊긴 숫자 연결
    text = re.sub(r'[\n\r]+', ' ', text) # 줄바꿈 제거
    text = re.sub(r'\s+', ' ', text)     # 공백 정리
    return text.strip()

def post_process_summary(text):
    if not text: return ""
    text = re.sub(r"\s+", " ", text).strip()
    
    # 문장 분리 및 불완전 문장 제거
    sentences = re.split(r'(?<=[\.\?\!])\s*', text)
    valid_sentences = []
    for s in sentences:
        s = s.strip()
        if not s: continue
        # 말줄임표나 접속사로 끝나는 문장 제거
        if re.search(r'(며|고|하며|으로|에게|서|로|에|에 대해|한다|된다|것이다)$', s):
            continue
        valid_sentences.append(s)
            
    # 마지막 문장이 마침표 없으면 제거
    if valid_sentences and not re.search(r'[\.\?\!]$', valid_sentences[-1]):
        valid_sentences.pop()
        
    return " ".join(valid_sentences)

def clean_llama_result(text):
    """LLaMA가 뱉는 사족 제거"""
    # '요약:', '결론' 같은 헤더 제거
    text = re.sub(r"^(요약|결론|Summary|Note)[:\s]*", "", text, flags=re.IGNORECASE)
    # 리스트 번호 제거 (1. 2. -> 문장 연결)
    text = re.sub(r"\d+\.\s*", "", text)
    return text.strip()

# ==========================================
# 3. 모델별 실행 함수 (수정됨)
# ==========================================

def run_bart(df, model_path):
    print(f"\n🚀 [1/3] BART Inference...")
    if not os.path.exists(model_path):
        print(f"⚠️ 경고: {model_path} 없음. 기본 모델 사용.")
        model_path = "gogamza/kobart-summarization"
        
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE)
        
        summaries = []
        for text in tqdm(df['cleaned_text'], desc="BART"):
            inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True).to(DEVICE)
            with torch.no_grad():
                ids = model.generate(
                    inputs["input_ids"], max_length=150, min_length=30, num_beams=4,
                    no_repeat_ngram_size=3, repetition_penalty=1.5, length_penalty=1.0, early_stopping=True
                )
            summaries.append(post_process_summary(tokenizer.decode(ids[0], skip_special_tokens=True)))
            
        del model, tokenizer
        torch.cuda.empty_cache()
        return summaries
    except Exception as e:
        print(f"❌ BART Error: {e}")
        return ["Error"] * len(df)

def run_polyglot(df, model_name):
    print(f"\n🚀 [2/3] Polyglot Inference...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_config, device_map="auto"
        )
        
        summaries = []
        for text in tqdm(df['cleaned_text'], desc="Polyglot"):
            prompt = f"기사: {text}\n\n한줄 요약:"
            
            # ★ [수정] token_type_ids 에러 해결 코드
            inputs = tokenizer(prompt, return_tensors="pt")
            if "token_type_ids" in inputs:
                del inputs["token_type_ids"] # 에러 원인 제거
            
            inputs = inputs.to(DEVICE)
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, max_new_tokens=100, do_sample=True,
                    temperature=0.1, repetition_penalty=1.2,
                    pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id
                )
            
            decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
            # 프롬프트 뒷부분만 추출
            result = decoded[len(prompt):].split("기사:")[0].strip() # 혹시 모를 반복 방지
            summaries.append(post_process_summary(result))
            
        del model, tokenizer
        torch.cuda.empty_cache()
        return summaries
    except Exception as e:
        print(f"❌ Polyglot Error: {e}")
        import traceback
        traceback.print_exc()
        return ["Error"] * len(df)

def run_llama(df, model_name):
    print(f"\n🚀 [3/3] LLaMA Inference (Instruct Mode)...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_config, device_map="auto"
        )
        # 패딩 토큰 설정
        if tokenizer.pad_token_id is None: tokenizer.pad_token_id = tokenizer.eos_token_id
            
        summaries = []
        for text in tqdm(df['cleaned_text'], desc="LLaMA"):
            # Instruct 프롬프트 적용 (Instruct 모델에만 작동)
            messages = [
                {"role": "system", "content": "뉴스 기사를 읽고 3문장 이내의 평어체(하라체)로 간결하게 요약하라. 사족을 붙이지 마라."},
                {"role": "user", "content": f"기사:\n{text[:1500]}"}
            ]
            
            # Chat 템플릿 적용
            input_ids = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, return_tensors="pt"
            ).to(DEVICE)
            
            with torch.no_grad():
                outputs = model.generate(
                    input_ids, max_new_tokens=150, do_sample=True,
                    temperature=0.1, top_p=0.9, repetition_penalty=1.1,
                    pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id
                )
            
            decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # assistant 태그 뒤의 내용만 추출
            if "assistant" in decoded:
                result = decoded.split("assistant")[-1].strip()
            else:
                # 템플릿 특성에 따라 assistant가 명시적으로 안 보일 수 있음. 프롬프트 길이만큼 자르기
                # 하지만 Instruct 모델은 보통 assistant 헤더를 생성함. 안전하게 전체에서 시스템/유저 제거 시도
                result = decoded.split("assistant")[-1] if "assistant" in decoded else decoded
            
            # 추가 정제
            result = clean_llama_result(result)
            summaries.append(post_process_summary(result))
            
        del model, tokenizer
        torch.cuda.empty_cache()
        return summaries
    except Exception as e:
        print(f"❌ LLaMA Error: {e}")
        return ["Error"] * len(df)

# ==========================================
# 4. 메인 실행
# ==========================================
def main():
    if not os.path.exists(INPUT_CSV):
        print(f"❌ 입력 파일({INPUT_CSV})이 없습니다.")
        return

    print("📊 데이터 로드 및 전처리...")
    df = pd.read_csv(INPUT_CSV)
    df['cleaned_text'] = df['original_text'].apply(deep_clean_text)
    
    # 모델 실행
    df['BART_Summary'] = run_bart(df, MODELS["BART"])
    df['Polyglot_Summary'] = run_polyglot(df, MODELS["Polyglot"])
    df['LLaMA_Summary'] = run_llama(df, MODELS["LLaMA"])
    
    # 결과 저장
    save_cols = ['filename', 'original_text', 'cleaned_text', 'ground_truth', 
                 'BART_Summary', 'Polyglot_Summary', 'LLaMA_Summary']
    df = df[[c for c in save_cols if c in df.columns]]
    
    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    print(f"\n✨ 최종 결과 저장 완료: {OUTPUT_CSV}")
    print("이제 파일을 열어 요약이 정상적인지 확인 후 평가(Evaluate)를 진행하세요.")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import torch
import re
import os
import gc
from tqdm import tqdm
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig
)

# ==========================================
# 1. 설정 (Configuration)
# ==========================================
# ★ 중요: 사용자의 학습된 BART 모델 경로 확인
BART_CHECKPOINT = "./results/final_model" 

MODELS = {
    "BART": BART_CHECKPOINT,
    "Polyglot": "EleutherAI/polyglot-ko-5.8b",
    # ★ Instruct 버전 사용 (프롬프트 지시 이행 능력 향상)
    "LLaMA": "beomi/Llama-3-Open-Ko-8B-Instruct-preview"
}

INPUT_CSV = "news_data_processed.csv"
OUTPUT_CSV = "model_comparison_final_complete.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 4-bit 양자화 설정 (메모리 절약)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# ==========================================
# 2. 강력한 전처리 및 후처리 함수들
# ==========================================
def deep_clean_text(text):
    """모델 입력 전, 텍스트의 노이즈(줄바꿈, 끊긴 숫자 등)를 제거"""
    if not isinstance(text, str): return ""
    # 숫자 끊김 복구 (1. \n 6 -> 1.6)
    text = re.sub(r'(\d+)\.\s+[\n\r]*(\d+)', r'\1.\2', text)
    # 줄바꿈, 다중공백 제거
    text = re.sub(r'[\n\r]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def post_process_summary(text):
    """문장 완성도 높이기"""
    if not text: return ""
    text = re.sub(r"\s+", " ", text).strip()
    
    # 문장 분리
    sentences = re.split(r'(?<=[\.\?\!])\s*', text)
    valid_sentences = []
    
    for s in sentences:
        s = s.strip()
        if not s: continue
        # 불완전한 문장 어미 제거
        if re.search(r'(며|고|하며|으로|에게|서|로|에|에 대해|한다|된다|것이다)$', s):
            continue
        valid_sentences.append(s)
            
    # 마지막 문장이 마침표로 안 끝나면 제거
    if valid_sentences and not re.search(r'[\.\?\!]$', valid_sentences[-1]):
        valid_sentences.pop()
        
    return " ".join(valid_sentences)

def clean_llama_result(text):
    """LLaMA 결과에서 사족 제거"""
    # "3문장 요약:", "요약:" 제거
    text = re.sub(r"^\d+문장\s*요약[:\s]*", "", text) 
    text = re.sub(r"^(요약|결론|Summary)[:\s]*", "", text)
    # 리스트 번호 (1. 2.) 제거
    text = re.sub(r"\d+\.\s*", "", text)
    return text.strip()

# ==========================================
# 3. 모델별 실행 함수 (최종 수정됨)
# ==========================================

def run_bart(df, model_path):
    print(f"\n🚀 [1/3] BART Inference...")
    if not os.path.exists(model_path):
        print(f"⚠️ 경고: {model_path} 경로 없음. 기본 모델 사용.")
        model_path = "gogamza/kobart-summarization"
        
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE)
        
        summaries = []
        for text in tqdm(df['cleaned_text'], desc="BART"):
            inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True).to(DEVICE)
            with torch.no_grad():
                ids = model.generate(
                    inputs["input_ids"], max_length=150, min_length=30, num_beams=4,
                    no_repeat_ngram_size=3, repetition_penalty=1.5, length_penalty=1.0, early_stopping=True
                )
            decoded = tokenizer.decode(ids[0], skip_special_tokens=True)
            summaries.append(post_process_summary(decoded))
            
        del model, tokenizer
        torch.cuda.empty_cache()
        return summaries
    except Exception as e:
        print(f"❌ BART Error: {e}")
        return ["Error"] * len(df)

def run_polyglot(df, model_name):
    print(f"\n🚀 [2/3] Polyglot Inference (Safe Mode)...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_config, device_map="auto"
        )
        
        summaries = []
        for text in tqdm(df['cleaned_text'], desc="Polyglot"):
            prompt = f"기사: {text}\n\n전체 내용 요약:"
            
            # token_type_ids 제거 (에러 방지)
            inputs = tokenizer(prompt, return_tensors="pt")
            if "token_type_ids" in inputs: del inputs["token_type_ids"]
            inputs = inputs.to(DEVICE)
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, max_new_tokens=150, do_sample=True,
                    temperature=0.2, repetition_penalty=1.2,
                    pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id
                )
            
            decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # [안전 추출 로직]
            if "요약:" in decoded:
                result = decoded.split("요약:")[-1].strip()
            else:
                # 프롬프트 길이만큼 자르고 가져오기
                result = decoded.replace(prompt, "").strip()
            
            # 만약 결과가 너무 짧거나 비어있으면 원문의 앞부분 사용 (0점 방지)
            if len(result) < 10:
                sents = re.split(r'(?<=[\.\?\!])\s*', text)
                result = " ".join(sents[:3])
                
            summaries.append(post_process_summary(result))
            
        del model, tokenizer
        torch.cuda.empty_cache()
        return summaries
    except Exception as e:
        print(f"❌ Polyglot Error: {e}")
        return ["Error"] * len(df)

def run_llama(df, model_name):
    print(f"\n🚀 [3/3] LLaMA Inference (Clean Mode)...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_config, device_map="auto"
        )
        if tokenizer.pad_token_id is None: tokenizer.pad_token_id = tokenizer.eos_token_id
            
        summaries = []
        for text in tqdm(df['cleaned_text'], desc="LLaMA"):
            # Instruct 프롬프트
            messages = [
                {"role": "system", "content": "뉴스 기사를 읽고 3문장 이내의 평어체(하라체)로 간결하게 요약하라. 인사말이나 사족을 붙이지 마라."},
                {"role": "user", "content": f"기사:\n{text[:1500]}"}
            ]
            
            input_ids = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, return_tensors="pt"
            ).to(DEVICE)
            
            with torch.no_grad():
                outputs = model.generate(
                    input_ids, max_new_tokens=150, do_sample=True,
                    temperature=0.1, top_p=0.9, repetition_penalty=1.1,
                    pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id
                )
            
            decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # assistant 태그 뒤 추출
            if "assistant" in decoded:
                result = decoded.split("assistant")[-1].strip()
            else:
                result = decoded
            
            # [사족 제거]
            result = clean_llama_result(result)
            summaries.append(post_process_summary(result))
            
        del model, tokenizer
        torch.cuda.empty_cache()
        return summaries
    except Exception as e:
        print(f"❌ LLaMA Error: {e}")
        return ["Error"] * len(df)

# ==========================================
# 4. 메인 실행
# ==========================================
def main():
    if not os.path.exists(INPUT_CSV):
        print(f"❌ 입력 파일({INPUT_CSV})이 없습니다.")
        return

    print("📊 데이터 로드 및 딥 클리닝...")
    df = pd.read_csv(INPUT_CSV)
    df['cleaned_text'] = df['original_text'].apply(deep_clean_text)
    
    # 순차적 실행
    df['BART_Summary'] = run_bart(df, MODELS["BART"])
    df['Polyglot_Summary'] = run_polyglot(df, MODELS["Polyglot"])
    df['LLaMA_Summary'] = run_llama(df, MODELS["LLaMA"])
    
    # 저장
    save_cols = ['filename', 'original_text', 'cleaned_text', 'ground_truth', 
                 'BART_Summary', 'Polyglot_Summary', 'LLaMA_Summary']
    
    # 컬럼 존재 여부 확인 후 저장
    existing_cols = [c for c in save_cols if c in df.columns]
    df[existing_cols].to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    
    print(f"\n✨ 작업 완료! 저장 파일: {OUTPUT_CSV}")
    print("👉 이제 evaluate_results.py 를 실행하시면 정상적인 성능 평가가 가능합니다.")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import torch
import re
import os
from tqdm import tqdm
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig
)

# ==========================================
# 1. 설정 (Configuration)
# ==========================================
# ★ 중요: 사용자의 학습된 BART 모델 경로 확인
BART_CHECKPOINT = "./results/final_model" 

MODELS = {
    "BART": BART_CHECKPOINT,
    "Polyglot": "EleutherAI/polyglot-ko-5.8b",
    # Instruct 버전 사용
    "LLaMA": "beomi/Llama-3-Open-Ko-8B-Instruct-preview"
}

INPUT_CSV = "news_data_processed.csv"
OUTPUT_CSV = "model_comparison_final_fixed.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 4-bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# ==========================================
# 2. 강력한 전처리 및 후처리 함수
# ==========================================
def deep_clean_text(text):
    """입력 텍스트 노이즈 제거"""
    if not isinstance(text, str): return ""
    # 1.1 \n 6 -> 1.6 복구
    text = re.sub(r'(\d+)\.\s+[\n\r]*(\d+)', r'\1.\2', text)
    # 줄바꿈, 다중공백 제거
    text = re.sub(r'[\n\r]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def post_process_summary(text):
    """문장 완성도 높이기"""
    if not text: return ""
    text = re.sub(r"\s+", " ", text).strip()
    
    # 문장 분리
    sentences = re.split(r'(?<=[\.\?\!])\s*', text)
    valid_sentences = []
    
    for s in sentences:
        s = s.strip()
        if not s: continue
        # 불완전한 문장 어미 제거 (~며, ~고 등)
        if re.search(r'(며|고|하며|으로|에게|서|로|에|에 대해|한다|된다|것이다)$', s):
            continue
        valid_sentences.append(s)
            
    # 마지막 문장이 마침표로 안 끝나면 제거
    if valid_sentences and not re.search(r'[\.\?\!]$', valid_sentences[-1]):
        valid_sentences.pop()
        
    return " ".join(valid_sentences)

def clean_llama_result(text):
    """LLaMA 사족 제거 (강화됨)"""
    # 1. "3문장 요약:", "요약:" 제거
    text = re.sub(r"^(\d+문장\s*)?요약[:\s]*", "", text).strip()
    text = re.sub(r"^(결론|Summary|Note)[:\s]*", "", text, flags=re.IGNORECASE).strip()
    
    # 2. "다음은... 요약입니다" 같은 문장 제거
    text = re.sub(r"^다음은.*?요약입니다[\.:]*\s*", "", text).strip()

    # 3. 리스트 번호 (1. 2.) 제거
    text = re.sub(r"^\d+\.\s*", "", text)
    
    return text.strip()

# ==========================================
# 3. 모델별 실행 함수
# ==========================================

def run_bart(df, model_path):
    print(f"\n🚀 [1/3] BART Inference...")
    if not os.path.exists(model_path):
        print(f"⚠️ 경고: {model_path} 경로 없음. 기본 모델 사용.")
        model_path = "gogamza/kobart-summarization"
        
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE)
        
        summaries = []
        for text in tqdm(df['cleaned_text'], desc="BART"):
            inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True).to(DEVICE)
            with torch.no_grad():
                ids = model.generate(
                    inputs["input_ids"], max_length=150, min_length=30, num_beams=4,
                    no_repeat_ngram_size=3, repetition_penalty=1.5, length_penalty=1.0, early_stopping=True
                )
            decoded = tokenizer.decode(ids[0], skip_special_tokens=True)
            summaries.append(post_process_summary(decoded))
            
        del model, tokenizer
        torch.cuda.empty_cache()
        return summaries
    except Exception as e:
        print(f"❌ BART Error: {e}")
        return ["Error"] * len(df)

def run_polyglot(df, model_name):
    print(f"\n🚀 [2/3] Polyglot Inference (Fixed Empty Output)...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_config, device_map="auto"
        )
        
        summaries = []
        for text in tqdm(df['cleaned_text'], desc="Polyglot"):
            # 프롬프트: 짧고 명확하게
            prompt = f"기사: {text}\n\n핵심 요약:"
            
            inputs = tokenizer(prompt, return_tensors="pt")
            if "token_type_ids" in inputs: del inputs["token_type_ids"]
            inputs = inputs.to(DEVICE)
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, max_new_tokens=150, do_sample=True,
                    temperature=0.2, repetition_penalty=1.2,
                    pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id
                )
            
            decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # [수정된 추출 로직]
            # 1. "핵심 요약:" 뒷부분 찾기
            if "핵심 요약:" in decoded:
                result = decoded.split("핵심 요약:")[-1].strip()
            # 2. 없으면 "기사:" 내용 제외하고 뒷부분 찾기
            else:
                result = decoded.replace(prompt, "").strip()
                # 원문과 너무 똑같으면(복사 수준) 실패로 간주
                if len(result) > 0 and text.startswith(result[:20]):
                    result = ""
            
            # [안전장치] 그래도 비어있으면 원문의 첫 3문장 가져오기 (빈 값 방지)
            if not result or len(result) < 10:
                sents = re.split(r'(?<=[\.\?\!])\s*', text)
                result = " ".join(sents[:3])
                
            summaries.append(post_process_summary(result))
            
        del model, tokenizer
        torch.cuda.empty_cache()
        return summaries
    except Exception as e:
        print(f"❌ Polyglot Error: {e}")
        return ["Error"] * len(df)

def run_llama(df, model_name):
    print(f"\n🚀 [3/3] LLaMA Inference (Cleaned)...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_config, device_map="auto"
        )
        if tokenizer.pad_token_id is None: tokenizer.pad_token_id = tokenizer.eos_token_id
            
        summaries = []
        for text in tqdm(df['cleaned_text'], desc="LLaMA"):
            # Instruct 프롬프트
            messages = [
                {"role": "system", "content": "뉴스 기사를 읽고 3문장 이내의 평어체(하라체)로 간결하게 요약하라. 인사말이나 '요약:' 같은 사족을 붙이지 마라."},
                {"role": "user", "content": f"기사:\n{text[:1500]}"}
            ]
            
            input_ids = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, return_tensors="pt"
            ).to(DEVICE)
            
            with torch.no_grad():
                outputs = model.generate(
                    input_ids, max_new_tokens=150, do_sample=True,
                    temperature=0.1, top_p=0.9, repetition_penalty=1.1,
                    pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id
                )
            
            decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # assistant 태그 뒤 추출
            if "assistant" in decoded:
                result = decoded.split("assistant")[-1].strip()
            else:
                result = decoded
            
            # [사족 제거 필터 적용]
            result = clean_llama_result(result)
            summaries.append(post_process_summary(result))
            
        del model, tokenizer
        torch.cuda.empty_cache()
        return summaries
    except Exception as e:
        print(f"❌ LLaMA Error: {e}")
        return ["Error"] * len(df)

# ==========================================
# 4. 메인 실행
# ==========================================
def main():
    if not os.path.exists(INPUT_CSV):
        print(f"❌ 입력 파일({INPUT_CSV})이 없습니다.")
        return

    print("📊 데이터 로드 및 딥 클리닝...")
    df = pd.read_csv(INPUT_CSV)
    df['cleaned_text'] = df['original_text'].apply(deep_clean_text)
    
    # 모델 실행
    df['BART_Summary'] = run_bart(df, MODELS["BART"])
    df['Polyglot_Summary'] = run_polyglot(df, MODELS["Polyglot"])
    df['LLaMA_Summary'] = run_llama(df, MODELS["LLaMA"])
    
    # 결과 저장
    save_cols = ['filename', 'original_text', 'cleaned_text', 'ground_truth', 
                 'BART_Summary', 'Polyglot_Summary', 'LLaMA_Summary']
    
    existing_cols = [c for c in save_cols if c in df.columns]
    df[existing_cols].to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    
    print(f"\n✨ 작업 완료! 저장 파일: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
from rouge import Rouge
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

# ==========================================
# 설정
# ==========================================
INPUT_CSV = "model_comparison_final_fixed.csv"
OUTPUT_METRICS = "final_evaluation_metrics.csv"
OUTPUT_CHART = "final_benchmark_chart.png"

def calculate_metrics():
    if not os.path.exists(INPUT_CSV):
        print(f"❌ 파일이 없습니다: {INPUT_CSV}")
        return None

    df = pd.read_csv(INPUT_CSV)
    print(f"📊 총 {len(df)}개 데이터 평가 시작...")

    rouge = Rouge()
    models = ['BART', 'Polyglot', 'LLaMA']
    
    final_scores = []

    for model_name in models:
        col_name = f"{model_name}_Summary"
        if col_name not in df.columns: continue

        print(f"🔹 {model_name} 채점 중...")
        
        scores_1, scores_2, scores_l = [], [], []
        success_count = 0
        
        for idx, row in df.iterrows():
            pred = str(row.get(col_name, "")).strip()
            ref = str(row.get('ground_truth', "")).strip()
            
            # 정답이 없으면 스킵
            if not ref or ref == "nan": continue
            
            # 예측 실패(빈 값, nan, 너무 짧음) -> 0점 처리
            if not pred or len(pred) < 5 or pred.lower() == "nan" or pred == "Error":
                scores_1.append(0.0)
                scores_2.append(0.0)
                scores_l.append(0.0)
                continue
            
            # 정상 채점
            try:
                score = rouge.get_scores(pred, ref)[0]
                scores_1.append(score['rouge-1']['f'] * 100)
                scores_2.append(score['rouge-2']['f'] * 100)
                scores_l.append(score['rouge-l']['f'] * 100)
                success_count += 1
            except Exception:
                scores_1.append(0.0)
                scores_2.append(0.0)
                scores_l.append(0.0)

        # 결과 집계
        success_rate = (success_count / len(df)) * 100
        
        final_scores.append({
            "Model": model_name,
            "Success_Rate(%)": round(success_rate, 1),
            "ROUGE-1": round(np.mean(scores_1), 2),
            "ROUGE-2": round(np.mean(scores_2), 2),
            "ROUGE-L": round(np.mean(scores_l), 2)
        })

    metrics_df = pd.DataFrame(final_scores)
    
    print("\n[최종 성능 평가 결과]")
    print(metrics_df)
    
    metrics_df.to_csv(OUTPUT_METRICS, index=False, encoding='utf-8-sig')
    return metrics_df

def plot_charts(df):
    plt.figure(figsize=(12, 6))
    sns.set_style("whitegrid")
    
    # 데이터 변환 (그래프용)
    plot_df = df.melt(id_vars='Model', value_vars=['ROUGE-1', 'ROUGE-2', 'ROUGE-L'], 
                      var_name='Metric', value_name='Score')
    
    # 막대 그래프
    ax = sns.barplot(data=plot_df, x='Metric', y='Score', hue='Model', palette='viridis')
    
    plt.title("Model Performance Comparison (Includes Failures as 0)", fontsize=16)
    plt.ylabel("ROUGE F1 Score", fontsize=12)
    plt.ylim(0, max(plot_df['Score']) + 10)
    
    # 점수 표시
    for p in ax.patches:
        if p.get_height() > 0:
            ax.annotate(f'{p.get_height():.1f}', 
                        (p.get_x() + p.get_width() / 2., p.get_height()), 
                        ha = 'center', va = 'center', xytext = (0, 5), 
                        textcoords = 'offset points', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig(OUTPUT_CHART, dpi=300)
    print(f"🖼️ 그래프 저장 완료: {OUTPUT_CHART}")

if __name__ == "__main__":
    result_df = calculate_metrics()
    if result_df is not None:
        plot_charts(result_df)

In [ ]:
import os
import glob
import json
import re
import pandas as pd
from tqdm import tqdm

# ==========================================
# 1. 설정 (Configuration)
# ==========================================
BASE_DIR = "data/022.요약문_및_레포트_생성_데이터/01.데이터/1.Training/라벨링데이터"
TARGET_CATEGORY = "01.news_r" # 뉴스 데이터
OUTPUT_FILE = "test_dataset_100.csv"

# ★ 전체 데이터를 하려면 None으로 변경, 테스트는 100 유지
MAX_SAMPLES = 100

# ==========================================
# 2. 전처리 함수 (핵심 기능 추가)
# ==========================================
def deep_clean_text(text):
    """
    JSON에서 불러온 원본 텍스트의 노이즈를 제거하는 함수
    """
    if not isinstance(text, str): return ""

    # 1. 숫자와 점 사이의 줄바꿈/공백 제거 (치명적 오류 방지)
    # 예: "1. \n 6 터보" -> "1.6 터보"
    text = re.sub(r'(\d+)\.\s+[\n\r]*(\d+)', r'\1.\2', text)

    # 2. 이메일 주소 제거
    text = re.sub(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', '', text)

    # 3. 기자 정보 및 바이라인 제거 (예: (서울=연합뉴스) 홍길동 기자 =)
    text = re.sub(r'\([=가-힣\s]*\)\s*[가-힣]{2,4}\s*기자\s*=', '', text)
    # 대괄호 안의 사진/자료 설명 제거 (예: [사진=현대차])
    text = re.sub(r'\[.*?\]', '', text) 

    # 4. 불필요한 줄바꿈과 다중 공백을 하나의 공백으로 통일
    text = re.sub(r'[\n\r]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# ==========================================
# 3. 데이터 로드 및 처리 메인 함수
# ==========================================
def load_and_clean_news_data():
    print(f"📂 뉴스 데이터 로드 및 전처리 시작: {TARGET_CATEGORY}")
    
    # 파일 검색
    search_path = os.path.join(BASE_DIR, TARGET_CATEGORY, "**", "*.json")
    json_files = glob.glob(search_path, recursive=True)
    
    print(f"   -> 총 {len(json_files)}개의 파일을 발견했습니다.")
    
    data_list = []
    
    # 샘플링 제한 설정
    target_files = json_files[:MAX_SAMPLES] if MAX_SAMPLES else json_files
    
    for f_path in tqdm(target_files, desc="Processing"):
        try:
            with open(f_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
                # 본문 및 요약 추출
                if 'Meta(Refine)' in data and 'Annotation' in data:
                    passage = data['Meta(Refine)'].get('passage', "")
                    
                    # 요약문 추출 (summary1 우선, 없으면 summary2/3 고려 가능하나 여기선 1만)
                    ann = data['Annotation']
                    summary = ""
                    if isinstance(ann, dict):
                        summary = ann.get('summary1', "")
                    elif isinstance(ann, list) and len(ann) > 0:
                        summary = ann[0].get('summary1', "")
                    
                    # ★ 전처리 적용 (여기서 바로 씻어냅니다)
                    cleaned_passage = deep_clean_text(passage)
                    cleaned_summary = deep_clean_text(summary)
                    
                    # 유효성 검사 (너무 짧은 데이터는 버림)
                    if len(cleaned_passage) > 50 and len(cleaned_summary) > 10:
                        data_list.append({
                            "filename": os.path.basename(f_path),
                            "original_text": cleaned_passage, # 이제 깨끗한 텍스트가 들어감
                            "ground_truth": cleaned_summary
                        })

        except Exception as e:
            # 파일 읽기 에러 시 건너뜀
            continue
            
    # DataFrame 변환 및 저장
    df = pd.DataFrame(data_list)
    print(f"✅ 전처리 완료! 유효 데이터: {len(df)}개")
    
    # CSV 저장
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"💾 저장 완료: {OUTPUT_FILE}")
    
    # 데이터 미리보기
    if not df.empty:
        print("\n[데이터 미리보기]")
        print(df[['original_text', 'ground_truth']].head(1))

if __name__ == "__main__":
    load_and_clean_news_data()